# **1. Cài đặt & Import thư viện (Import Lib)**

In [1]:
!pip install gensim

import pandas as pd
import numpy as np
import os
import pickle
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC # Import Support Vector Classifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 23.0 MB/s eta 0:00:00
Mounted at /content/drive


# **2. Chuẩn bị dữ liệu (Prepare Data)**

In [2]:
path_project = "/content/drive/MyDrive/FakeNewsDetection_Project"
path_dataset = os.path.join(path_project, "Dataset")

df_true = pd.read_csv(os.path.join(path_dataset, "True.csv"))
df_fake = pd.read_csv(os.path.join(path_dataset, "Fake.csv"))

df_true['label'] = 1
df_fake['label'] = 0

df = pd.concat([df_true, df_fake], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df[['text', 'label']].dropna()

/tmp/ipykernel_2654/1950524972.py:5: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fake = pd.read_csv(os.path.join(path_dataset, "Fake.csv"))


# **3. Chia dữ liệu (Prepare Training Data - 80/20)**

In [3]:
df['tokenized_text'] = df['text'].apply(lambda x: simple_preprocess(str(x)))

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['tokenized_text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

# **4. Huấn luyện Word2Vec (Word2Vec Model)**

In [4]:
# Bạn có thể dùng lại Model Word2Vec đã train ở bài trước để tiết kiệm thời gian
w2v_model = Word2Vec(sentences=X_train_raw, vector_size=100, window=5, min_count=2, workers=4)

def get_avg_vec(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv.key_to_index]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X_train = np.array([get_avg_vec(text, w2v_model) for text in X_train_raw])
X_test = np.array([get_avg_vec(text, w2v_model) for text in X_test_raw])

# **5. Huấn luyện SVM (Training with SVM)**

In [5]:
# Khởi tạo SVM với kernel tuyến tính (thường dùng cho text)
svm_model = SVC(kernel='linear', C=1.0, random_state=42)

# Huấn luyện
print("Đang huấn luyện SVM... (Vui lòng đợi)")
svm_model.fit(X_train, y_train)

# Dự đoán
y_pred = svm_model.predict(X_test)

# Đánh giá APRF
metrics = {
    'acc': accuracy_score(y_test, y_pred),
    'pre': precision_score(y_test, y_pred),
    'rec': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

print("\n=== KẾT QUẢ SVM (APRF) ===")
print(f"Accuracy:  {metrics['acc']:.4f}")
print(f"Precision: {metrics['pre']:.4f}")
print(f"Recall:    {metrics['rec']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")

Đang huấn luyện SVM... (Vui lòng đợi)

=== KẾT QUẢ SVM (APRF) ===
Accuracy:  0.9667
Precision: 0.9625
Recall:    0.9685
F1-Score:  0.9655


# **6. Lưu Model & Cập nhật Metadata (Bonus)**

In [6]:
path_models = os.path.join(path_project, "Models")

# Lưu file .pkl của SVM
with open(os.path.join(path_models, "svm_classifier.pkl"), 'wb') as f:
    pickle.dump(svm_model, f)

# Ghi chú vào file text Metadata
metadata_path = os.path.join(path_models, "model_info.txt")
with open(metadata_path, "a", encoding="utf-8") as f:
    f.write(f"\n- svm_classifier.pkl: Accuracy {metrics['acc']:.4f}, dùng Word2Vec 100D, Kernel Linear.\n")

print("Đã lưu mô hình SVM và cập nhật ghi chú!")

Đã lưu mô hình SVM và cập nhật ghi chú!
